# Proyeksi Outflow dengan SARIMA — Google Colab

Jalankan sel **dari atas ke bawah** (Runtime → Run all, atau Shift+Enter per sel).

Isi notebook:
1. Menyiapkan mesin model
2. Mengunggah data
3. Pengaturan periode dan model
4. Membaca & memeriksa data
5. Mencocokkan hasil dengan EViews
6. Diagnostik residual
7. Backtest (data uji)
8. Proyeksi ke depan + unduh Excel
9. *(Opsional)* Menjalankan tampilan web Streamlit dari Colab

Model diestimasi pada **log(outflow)**; hasil dikembalikan ke skala Rupiah. Konstanta **C** diperlakukan sebagai rata-rata, sama seperti EViews.

## 1. Menyiapkan mesin model
Sel ini menulis file `arima_engine.py` di Colab. Cukup dijalankan, tidak perlu diubah.

In [ ]:
%%writefile arima_engine.py
"""
Mesin proyeksi SARIMA untuk outflow (dimodelkan dalam log), disusun agar
sedekat mungkin dengan pengaturan EViews:

- Y = log(outflow)
- Konstanta "C" diperlakukan seperti EViews: sebagai RATA-RATA (regresi pada
  konstanta dengan galat ARMA), bukan intercept persamaan AR.
- Estimasi Maximum Likelihood, kovarians koefisien dengan outer product of
  gradients (OPG) seperti keluaran EViews.
- Kriteria informasi (AIC/SC/HQ) dilaporkan per observasi, seperti EViews.

Modul ini tidak bergantung pada Streamlit, jadi bisa dipakai dari skrip biasa.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats

# ---------------------------------------------------------------------------
# 1. Membaca dan memvalidasi data
# ---------------------------------------------------------------------------

_POLA_EVIEWS = re.compile(r"^\s*(\d{4})\s*[Mm]\s*(\d{1,2})\s*$")  # contoh: 2021M01


def baca_file(file) -> pd.DataFrame:
    """Baca CSV atau Excel. `file` boleh path (str) atau objek unggahan Streamlit."""
    nama = getattr(file, "name", str(file)).lower()
    if nama.endswith((".xlsx", ".xls")):
        return pd.read_excel(file)
    # CSV: coba deteksi pemisah (koma atau titik koma)
    return pd.read_csv(file, sep=None, engine="python")


def ke_periode_bulanan(kolom: pd.Series) -> pd.PeriodIndex:
    """Ubah kolom tanggal (2021M01, 2021-01, 01/01/2021, tanggal Excel) ke periode bulanan."""
    teks = kolom.astype(str).str.strip()
    if teks.str.match(_POLA_EVIEWS).all():
        bagian = teks.str.extract(_POLA_EVIEWS)
        label = bagian[0] + "-" + bagian[1].str.zfill(2)
        return pd.PeriodIndex(label, freq="M")
    tanggal = pd.to_datetime(kolom, errors="raise")
    return pd.PeriodIndex(tanggal, freq="M")


def _ke_angka(kolom: pd.Series) -> pd.Series:
    """Ubah teks angka ke float. Menangani format Indonesia (1.234.567,89)."""
    if pd.api.types.is_numeric_dtype(kolom):
        return kolom.astype(float)
    teks = kolom.astype(str).str.strip()
    format_id = teks.str.contains(r",\d+$") | teks.str.contains(r"\.\d{3}(?:\.|$)")
    teks = teks.where(~format_id, teks.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(teks, errors="coerce")


@dataclass
class HasilValidasi:
    seri: pd.Series                       # seri bulanan (level, bukan log)
    galat: list[str] = field(default_factory=list)       # menghentikan proses
    peringatan: list[str] = field(default_factory=list)  # boleh lanjut

    @property
    def ok(self) -> bool:
        return not self.galat


def siapkan_seri(df: pd.DataFrame, kolom_tanggal: str, kolom_nilai: str) -> HasilValidasi:
    """Bentuk seri bulanan dan periksa masalah umum pada data."""
    galat, peringatan = [], []
    try:
        periode = ke_periode_bulanan(df[kolom_tanggal])
    except Exception as e:  # format tanggal tidak dikenali
        return HasilValidasi(pd.Series(dtype=float), [f"Kolom tanggal tidak terbaca: {e}"])

    nilai = _ke_angka(df[kolom_nilai])
    seri = pd.Series(nilai.to_numpy(), index=periode, name=kolom_nilai).sort_index()

    if seri.index.duplicated().any():
        dup = sorted({str(p) for p in seri.index[seri.index.duplicated()]})
        galat.append(f"Ada bulan ganda: {', '.join(dup)}")
        seri = seri[~seri.index.duplicated(keep="first")]

    lengkap = pd.period_range(seri.index.min(), seri.index.max(), freq="M")
    hilang = lengkap.difference(seri.index)
    if len(hilang):
        peringatan.append(
            f"Ada {len(hilang)} bulan yang tidak ada di data: "
            + ", ".join(str(p) for p in hilang[:12]) + (" ..." if len(hilang) > 12 else "")
            + ". Bulan ini dianggap kosong (missing)."
        )
    seri = seri.reindex(lengkap)

    kosong = seri.isna() & ~seri.index.isin(hilang)
    if kosong.any():
        peringatan.append(f"Ada {int(kosong.sum())} nilai kosong/tidak terbaca sebagai angka.")

    tidak_positif = seri <= 0
    if tidak_positif.any():
        bulan = ", ".join(str(p) for p in seri.index[tidak_positif][:12])
        galat.append(f"Nilai harus > 0 karena akan di-log. Bermasalah di: {bulan}")

    if seri.notna().sum() < 36:
        peringatan.append("Data kurang dari 36 bulan; pola musiman 12 bulan sulit diestimasi dengan andal.")

    return HasilValidasi(seri, galat, peringatan)


# ---------------------------------------------------------------------------
# 2. Spesifikasi dan estimasi model
# ---------------------------------------------------------------------------

@dataclass
class SpesifikasiModel:
    order: tuple[int, int, int] = (0, 0, 0)
    seasonal_order: tuple[int, int, int, int] = (1, 0, 1, 12)
    konstanta: bool = True
    # EViews tidak memaksa stasioner/invertible, jadi default False agar
    # hasilnya bisa dibandingkan. Ubah ke True untuk estimasi yang "aman".
    paksa_stasioner: bool = False
    paksa_invertible: bool = False

    def label(self) -> str:
        p, d, q = self.order
        P, D, Q, s = self.seasonal_order
        return f"SARIMA({p},{d},{q})({P},{D},{Q}){s}" + (" + C" if self.konstanta else "")


def _exog_konstanta(index, pakai: bool):
    if not pakai:
        return None
    return pd.DataFrame({"C": np.ones(len(index))}, index=index)


def estimasi(log_y: pd.Series, spek: SpesifikasiModel):
    """Estimasi SARIMA dengan ML. Mengembalikan objek hasil statsmodels."""
    from statsmodels.tsa.statespace.sarimax import SARIMAX  # impor di sini agar modul tetap bisa dimuat tanpa statsmodels

    model = SARIMAX(
        log_y,
        exog=_exog_konstanta(log_y.index, spek.konstanta),
        order=spek.order,
        seasonal_order=spek.seasonal_order,
        trend="n",
        enforce_stationarity=spek.paksa_stasioner,
        enforce_invertibility=spek.paksa_invertible,
    )
    return model.fit(disp=False, maxiter=2000, cov_type="opg")


_NAMA_EVIEWS = {"sigma2": "SIGMASQ"}


def _nama_parameter(nama: str) -> str:
    if nama in _NAMA_EVIEWS:
        return _NAMA_EVIEWS[nama]
    m = re.match(r"^(ar|ma)\.(S\.)?L(\d+)$", nama)
    if m:
        jenis, musiman, lag = m.groups()
        return f"{'S' if musiman else ''}{jenis.upper()}({lag})"
    return nama


def tabel_koefisien(hasil) -> pd.DataFrame:
    tabel = pd.DataFrame({
        "Koefisien": hasil.params,
        "Std. Error": hasil.bse,
        "z-Statistic": hasil.tvalues,
        "Prob.": hasil.pvalues,
    })
    tabel.index = [_nama_parameter(n) for n in tabel.index]
    tabel.index.name = "Variabel"
    return tabel


def _residual_efektif(hasil) -> pd.Series:
    """Residual tanpa periode awal (burn-in) inisialisasi difus, jika ada."""
    burn = int(getattr(hasil, "loglikelihood_burn", 0) or 0)
    return pd.Series(hasil.resid).iloc[burn:].dropna()


def statistik_ringkas(hasil, log_y: pd.Series) -> pd.Series:
    """Statistik ringkas bergaya EViews (kriteria informasi per observasi)."""
    n = int(hasil.nobs)
    k = len(hasil.params)  # EViews ikut menghitung SIGMASQ
    ll = float(hasil.llf)
    resid = _residual_efektif(hasil)
    y_sejajar = log_y.loc[resid.index] if isinstance(resid.index, pd.PeriodIndex) else log_y.iloc[-len(resid):]
    ssr = float((resid ** 2).sum())
    sst = float(((y_sejajar - y_sejajar.mean()) ** 2).sum())
    return pd.Series({
        "Jumlah observasi": n,
        "Log likelihood": ll,
        "Akaike info criterion": (-2 * ll + 2 * k) / n,
        "Schwarz criterion": (-2 * ll + k * np.log(n)) / n,
        "Hannan-Quinn criter.": (-2 * ll + 2 * k * np.log(np.log(n))) / n,
        "R-squared (perkiraan)": 1 - ssr / sst if sst > 0 else np.nan,
        "Sum squared resid": ssr,
        "Observasi burn-in (dikecualikan dari residual)": int(getattr(hasil, "loglikelihood_burn", 0) or 0),
    })


# ---------------------------------------------------------------------------
# 3. Diagnostik
# ---------------------------------------------------------------------------

def diagnostik(hasil, spek: SpesifikasiModel) -> dict:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    from statsmodels.tsa.stattools import acf

    resid = _residual_efektif(hasil)
    p, _, q = spek.order
    P, _, Q, _ = spek.seasonal_order
    df_model = p + q + P + Q
    lag_uji = [lag for lag in (12, 24) if lag < len(resid)]
    ljung_box = acorr_ljungbox(resid, lags=lag_uji, model_df=df_model) if lag_uji else pd.DataFrame()

    jb = stats.jarque_bera(resid)
    nlags = min(24, len(resid) - 1)
    nilai_acf = acf(resid, nlags=nlags, fft=False)[1:]
    batas_acf = 1.96 / np.sqrt(len(resid))

    akar_ar = 1 / np.abs(hasil.arroots) if len(hasil.arroots) else np.array([])
    akar_ma = 1 / np.abs(hasil.maroots) if len(hasil.maroots) else np.array([])

    return {
        "ljung_box": ljung_box,
        "jarque_bera": {"statistik": float(jb.statistic), "p_value": float(jb.pvalue)},
        "acf": pd.DataFrame({"lag": np.arange(1, nlags + 1), "acf": nilai_acf}),
        "batas_acf": batas_acf,
        "modulus_akar_ar_invers": akar_ar,
        "modulus_akar_ma_invers": akar_ma,
        "peringatan": peringatan_model(hasil, akar_ar, akar_ma),
    }


def peringatan_model(hasil, akar_ar, akar_ma, ambang: float = 0.99) -> list[str]:
    """Deteksi masalah yang membuat model tidak bisa ditafsirkan seperti ARIMA biasa."""
    pesan = []
    if len(akar_ar) and akar_ar.max() >= ambang:
        pesan.append(
            f"Akar AR invers = {akar_ar.max():.4f} (≥ {ambang}). Komponen AR berada di/dekat unit root, "
            "sehingga model tidak stasioner. Nilai C (rata-rata) praktis tidak teridentifikasi."
        )
    if len(akar_ma) and akar_ma.max() >= ambang:
        pesan.append(
            f"Akar MA invers = {akar_ma.max():.4f} (≥ {ambang}). Komponen MA tidak/nyaris tidak invertible."
        )
    params = hasil.params
    for ar, ma in (("ar.S.L12", "ma.S.L12"), ("ar.L1", "ma.L1")):
        if ar in params.index and ma in params.index:
            phi, theta = params[ar], params[ma]
            if abs(phi + theta) < 0.05 and abs(phi) > 0.5:
                pesan.append(
                    f"{_nama_parameter(ar)} = {phi:.4f} dan {_nama_parameter(ma)} = {theta:.4f} hampir saling "
                    "meniadakan (pembatalan akar). Model ini pada dasarnya setara dengan pola musiman "
                    "deterministik (rata-rata per bulan) + white noise; t-statistik yang sangat besar "
                    "tidak bermakna."
                )
    se = hasil.bse
    if (se.drop("sigma2", errors="ignore") < 1e-3).any():
        pesan.append("Ada standard error yang sangat kecil (< 0,001): tanda estimasi berada di batas ruang parameter.")
    return pesan


# ---------------------------------------------------------------------------
# 4. Ukuran akurasi dan baseline
# ---------------------------------------------------------------------------

def ukuran_akurasi(aktual, ramalan) -> dict:
    a = np.asarray(aktual, dtype=float)
    f = np.asarray(ramalan, dtype=float)
    galat = a - f
    hasil = {
        "MAPE (%)": float(np.mean(np.abs(galat / a)) * 100),
        "sMAPE (%)": float(np.mean(np.abs(galat) / ((np.abs(a) + np.abs(f)) / 2)) * 100),
        "MAE": float(np.mean(np.abs(galat))),
        "RMSE": float(np.sqrt(np.mean(galat ** 2))),
    }
    if len(a) > 1:
        pembilang = np.sum(((f[1:] - a[1:]) / a[:-1]) ** 2)
        penyebut = np.sum(((a[1:] - a[:-1]) / a[:-1]) ** 2)
        hasil["Theil U2"] = float(np.sqrt(pembilang / penyebut)) if penyebut > 0 else np.nan
    return hasil


def baseline_naif_musiman(y_latih: pd.Series, periode_uji: pd.PeriodIndex) -> pd.Series:
    """Ramalan = nilai bulan yang sama setahun sebelumnya (hanya dari data latih)."""
    gabung = y_latih.copy()
    for p in periode_uji:
        gabung.loc[p] = gabung.get(p - 12, np.nan)
    return gabung.loc[periode_uji].rename("Naif musiman")


def baseline_rata_bulanan(y_latih: pd.Series, periode_uji: pd.PeriodIndex) -> pd.Series:
    """Ramalan = rata-rata geometrik (rata-rata log) per bulan kalender di data latih."""
    rata = np.log(y_latih.dropna()).groupby(y_latih.dropna().index.month).mean()
    return pd.Series(np.exp(rata.reindex(periode_uji.month).to_numpy()), index=periode_uji,
                     name="Rata-rata per bulan")


# ---------------------------------------------------------------------------
# 5. Proyeksi dan backtest
# ---------------------------------------------------------------------------

def proyeksi(hasil, langkah: int, spek: SpesifikasiModel, alpha: float = 0.05,
             koreksi_bias: bool = False) -> pd.DataFrame:
    """Proyeksi dinamis `langkah` bulan ke depan, dikembalikan ke skala Rupiah (exp)."""
    akhir = pd.Period(hasil.fittedvalues.index[-1], "M")
    index_depan = pd.period_range(akhir + 1, periods=langkah, freq="M")
    ramalan = hasil.get_forecast(steps=langkah, exog=_exog_konstanta(index_depan, spek.konstanta))
    rata_log = np.asarray(ramalan.predicted_mean)
    se_log = np.asarray(ramalan.se_mean)
    ci = np.asarray(ramalan.conf_int(alpha=alpha))
    faktor = np.exp(se_log ** 2 / 2) if koreksi_bias else 1.0
    return pd.DataFrame({
        "log_proyeksi": rata_log,
        "se_log": se_log,
        "proyeksi": np.exp(rata_log) * faktor,
        "batas_bawah": np.exp(ci[:, 0]),
        "batas_atas": np.exp(ci[:, 1]),
    }, index=index_depan)


def potong(seri: pd.Series, awal, akhir) -> pd.Series:
    return seri.loc[pd.Period(awal, "M"):pd.Period(akhir, "M")]


def backtest(y: pd.Series, spek: SpesifikasiModel, awal_latih, akhir_latih, akhir_uji,
             alpha: float = 0.05, koreksi_bias: bool = False) -> dict:
    """Estimasi di data latih, ramal dinamis sepanjang periode uji, bandingkan dengan aktual."""
    y_latih = potong(y, awal_latih, akhir_latih)
    y_uji = potong(y, pd.Period(akhir_latih, "M") + 1, akhir_uji)
    hasil = estimasi(np.log(y_latih), spek)
    ramal = proyeksi(hasil, len(y_uji), spek, alpha, koreksi_bias)

    tabel = pd.DataFrame({
        "aktual": y_uji,
        spek.label(): ramal["proyeksi"],
        "batas_bawah": ramal["batas_bawah"],
        "batas_atas": ramal["batas_atas"],
        "Naif musiman": baseline_naif_musiman(y_latih, y_uji.index),
        "Rata-rata per bulan": baseline_rata_bulanan(y_latih, y_uji.index),
    })
    sah = tabel["aktual"].notna()
    akurasi = pd.DataFrame({
        nama: ukuran_akurasi(tabel.loc[sah, "aktual"], tabel.loc[sah, nama])
        for nama in (spek.label(), "Naif musiman", "Rata-rata per bulan")
    }).T
    return {"hasil": hasil, "tabel": tabel, "akurasi": akurasi}

## 2. Mengunggah data
File CSV/Excel berisi dua kolom: tanggal (`2021M01`, `2021-01`, atau tanggal Excel) dan outflow dalam **level** (belum di-log).

Klik **Choose Files** setelah sel dijalankan. Kalau belum punya file, lewati sel ini dan jalankan sel berikutnya untuk memakai data contoh sintetis.

In [ ]:
from google.colab import files
unggahan = files.upload()
NAMA_FILE = list(unggahan)[0]
print("File:", NAMA_FILE)

In [ ]:
# Data contoh SINTETIS (bukan data asli) — hanya dipakai bila belum ada file yang diunggah
import numpy as np, pandas as pd
if "NAMA_FILE" not in globals():
    rng = np.random.default_rng(7)
    per = pd.period_range("2021-01", "2026-06", freq="M")
    pola = {1:-0.6, 2:-0.3, 3:0.4, 4:0.6, 5:0.3, 6:0.0, 7:-0.1, 8:0.0, 9:-0.2, 10:-0.1, 11:0.1, 12:0.7}
    log_y = 12.6 + np.array([pola[p.month] for p in per]) + rng.normal(0, 0.45, len(per))
    pd.DataFrame({"tanggal": [f"{p.year}M{p.month:02d}" for p in per],
                  "outflow": np.round(np.exp(log_y))}).to_csv("contoh_data.csv", index=False)
    NAMA_FILE = "contoh_data.csv"
    print("Memakai data contoh SINTETIS:", NAMA_FILE)

## 3. Pengaturan
Ubah nilai di sel ini sesuai data Anda, lalu jalankan.

In [ ]:
KOLOM_TANGGAL = "tanggal"     # nama kolom tanggal di file Anda
KOLOM_NILAI   = "outflow"     # nama kolom outflow (level, belum di-log)

AWAL_LATIH  = "2021-01"
AKHIR_LATIH = "2025-06"
AKHIR_UJI   = "2026-06"

ORDER          = (0, 0, 0)        # (p, d, q)
SEASONAL_ORDER = (1, 0, 1, 12)    # (P, D, Q, s) -> model EViews (0,0)(1,1)
KONSTANTA      = True

PAKSA_STASIONER  = False   # EViews tidak memaksa; ubah ke True bila AR lari ke 1
PAKSA_INVERTIBLE = False
KOREKSI_BIAS     = False   # exp(proyeksi log) * exp(sigma^2/2); EViews default: tidak

HORIZON   = 12     # bulan ke depan
TINGKAT   = 95     # selang kepercayaan (%)

## 4. Membaca & memeriksa data

In [ ]:
import importlib, numpy as np, pandas as pd
import plotly.graph_objects as go
import arima_engine as eng
importlib.reload(eng)

v = eng.siapkan_seri(eng.baca_file(NAMA_FILE), KOLOM_TANGGAL, KOLOM_NILAI)
for p in v.peringatan: print("PERINGATAN:", p)
if not v.ok: raise ValueError("\n".join(v.galat))
y = v.seri
print(f"{y.notna().sum()} bulan: {y.index.min()} – {y.index.max()}")

spek = eng.SpesifikasiModel(order=ORDER, seasonal_order=SEASONAL_ORDER, konstanta=KONSTANTA,
                            paksa_stasioner=PAKSA_STASIONER, paksa_invertible=PAKSA_INVERTIBLE)
alpha = 1 - TINGKAT / 100

fig = go.Figure(go.Scatter(x=y.index.to_timestamp(), y=y, mode="lines+markers", name="Outflow"))
fig.update_layout(height=350, title="Outflow bulanan", yaxis_title="Rupiah")
fig.show()

## 5. Estimasi & pencocokan dengan EViews
Kolom **Selisih** menunjukkan seberapa jauh hasil Python dari EViews. Selisih kecil wajar; selisih besar berarti pengaturan belum sama.

In [ ]:
log_latih = np.log(eng.potong(y, AWAL_LATIH, AKHIR_LATIH))
hasil = eng.estimasi(log_latih, spek)
print(spek.label(), "| observasi:", len(log_latih))
display(eng.tabel_koefisien(hasil).round(6))

EVIEWS = {"C": 12.67966, "SAR(12)": 1.000000, "SMA(12)": -0.999895, "SIGMASQ": 0.403474,
          "Log likelihood": -60.83680, "Akaike info criterion": 2.401363,
          "Schwarz criterion": 2.548695, "Hannan-Quinn criter.": 2.458183}
py = {**eng.tabel_koefisien(hasil)["Koefisien"].to_dict(), **eng.statistik_ringkas(hasil, log_latih).to_dict()}
banding = pd.DataFrame({"EViews": EVIEWS, "Python": {k: py.get(k, np.nan) for k in EVIEWS}})
banding["Selisih"] = banding["Python"] - banding["EViews"]
display(banding.round(6))
display(eng.statistik_ringkas(hasil, log_latih).to_frame("Nilai").round(6))

## 6. Diagnostik residual

In [ ]:
diag = eng.diagnostik(hasil, spek)
for p in diag["peringatan"]: print("⚠️", p, "\n")
print("Ljung-Box (p > 0,05 = tidak ada autokorelasi tersisa):")
display(diag["ljung_box"])
jb = diag["jarque_bera"]
print(f"Jarque-Bera = {jb['statistik']:.3f}, p = {jb['p_value']:.4f}")
print("Modulus akar invers AR maks:", np.round(diag["modulus_akar_ar_invers"].max(), 4) if len(diag["modulus_akar_ar_invers"]) else "-")
print("Modulus akar invers MA maks:", np.round(diag["modulus_akar_ma_invers"].max(), 4) if len(diag["modulus_akar_ma_invers"]) else "-")

a = diag["acf"]
fig = go.Figure(go.Bar(x=a["lag"], y=a["acf"]))
for t in (1, -1): fig.add_hline(y=t * diag["batas_acf"], line_dash="dash", line_color="gray")
fig.update_layout(height=300, title="ACF residual")
fig.show()

## 7. Backtest
Model diestimasi pada data latih, lalu diramal dinamis sepanjang data uji dan dibandingkan dengan dua baseline: **naif musiman** (nilai bulan yang sama tahun lalu) dan **rata-rata per bulan**.

In [ ]:
bt = eng.backtest(y, spek, AWAL_LATIH, AKHIR_LATIH, AKHIR_UJI, alpha, KOREKSI_BIAS)
tabel = bt["tabel"]; label = spek.label(); x = tabel.index.to_timestamp()

fig = go.Figure()
fig.add_scatter(x=x, y=tabel["batas_atas"], line=dict(width=0), showlegend=False)
fig.add_scatter(x=x, y=tabel["batas_bawah"], fill="tonexty", line=dict(width=0),
                fillcolor="rgba(31,119,180,0.15)", name=f"Selang {TINGKAT}%")
fig.add_scatter(x=x, y=tabel["aktual"], name="Aktual", mode="lines+markers", line=dict(color="black"))
fig.add_scatter(x=x, y=tabel[label], name=label, mode="lines+markers")
fig.add_scatter(x=x, y=tabel["Naif musiman"], name="Naif musiman", line=dict(dash="dot"))
fig.add_scatter(x=x, y=tabel["Rata-rata per bulan"], name="Rata-rata per bulan", line=dict(dash="dash"))
fig.update_layout(height=420, title="Backtest", yaxis_title="Rupiah")
fig.show()

display(bt["akurasi"].round(2))
display(tabel.rename(index=str).round(0))

## 8. Proyeksi ke depan
Model diestimasi ulang memakai seluruh data (sampai bulan terakhir), lalu diproyeksikan `HORIZON` bulan ke depan.

In [ ]:
AKHIR_DATA_PROYEKSI = str(y.dropna().index.max())   # ubah bila ingin memotong data
y_proj = eng.potong(y, AWAL_LATIH, AKHIR_DATA_PROYEKSI)
hasil_proj = eng.estimasi(np.log(y_proj), spek)
for p in eng.diagnostik(hasil_proj, spek)["peringatan"]: print("⚠️", p, "\n")
ramalan = eng.proyeksi(hasil_proj, HORIZON, spek, alpha, KOREKSI_BIAS)

fig = go.Figure()
fig.add_scatter(x=y_proj.index.to_timestamp(), y=y_proj, name="Historis", line=dict(color="black"))
xr = ramalan.index.to_timestamp()
fig.add_scatter(x=xr, y=ramalan["batas_atas"], line=dict(width=0), showlegend=False)
fig.add_scatter(x=xr, y=ramalan["batas_bawah"], fill="tonexty", line=dict(width=0),
                fillcolor="rgba(31,119,180,0.15)", name=f"Selang {TINGKAT}%")
fig.add_scatter(x=xr, y=ramalan["proyeksi"], name="Proyeksi", mode="lines+markers")
fig.update_layout(height=420, title=f"Proyeksi {HORIZON} bulan — {spek.label()}", yaxis_title="Rupiah")
fig.show()

keluaran = ramalan[["proyeksi", "batas_bawah", "batas_atas"]].rename(index=str)
keluaran.index.name = "periode"
display(keluaran.round(0))

In [ ]:
# Unduh hasil ke laptop (Excel: proyeksi, koefisien, akurasi backtest)
with pd.ExcelWriter("proyeksi_outflow.xlsx") as w:
    keluaran.to_excel(w, sheet_name="proyeksi")
    eng.tabel_koefisien(hasil_proj).to_excel(w, sheet_name="koefisien")
    bt["akurasi"].to_excel(w, sheet_name="akurasi_backtest")
    bt["tabel"].rename(index=str).to_excel(w, sheet_name="backtest")
from google.colab import files
files.download("proyeksi_outflow.xlsx")

## 9. *(Opsional)* Tampilan web Streamlit dari Colab

Bagian ini menjalankan aplikasi web yang sama dan memberi **tautan sementara** (`...trycloudflare.com`) yang bisa dibuka di browser.

⚠️ Tautan ini **publik** selama sel berjalan: siapa pun yang tahu tautannya bisa membuka aplikasi dan data yang Anda unggah ke dalamnya. Jangan bagikan tautannya bila datanya rahasia. Untuk menutupnya: menu **Runtime → Disconnect and delete runtime**. Kalau tautan belum muncul, tunggu beberapa detik lalu jalankan ulang sel terakhir.

In [ ]:
!pip -q install streamlit

In [ ]:
%%writefile app.py
"""
Aplikasi web proyeksi outflow dengan SARIMA (log).
Jalankan:  streamlit run app.py
"""
from __future__ import annotations

import io
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

import arima_engine as eng

st.set_page_config(page_title="Proyeksi Outflow — SARIMA", layout="wide")
st.title("Proyeksi Outflow dengan SARIMA")
st.caption("Model diestimasi pada log(outflow); hasil dikembalikan ke skala Rupiah.")

FILE_CONTOH = Path(__file__).with_name("contoh_data.csv")

# ---------------------------------------------------------------------------
# Sidebar: data dan pengaturan
# ---------------------------------------------------------------------------
with st.sidebar:
    st.header("1. Data")
    unggahan = st.file_uploader("Unggah CSV/Excel", type=["csv", "xlsx", "xls"])
    if unggahan is None:
        st.info("Belum ada file. Memakai data contoh (SINTETIS, bukan data asli).")
        sumber = FILE_CONTOH
    else:
        sumber = unggahan

try:
    df_mentah = eng.baca_file(sumber)
except Exception as e:
    st.error(f"File tidak bisa dibaca: {e}")
    st.stop()

kolom = list(df_mentah.columns)
with st.sidebar:
    tebak_tgl = next((c for c in kolom if str(c).lower() in ("tanggal", "periode", "date", "bulan")), kolom[0])
    tebak_nilai = next((c for c in kolom if "outflow" in str(c).lower()), kolom[min(1, len(kolom) - 1)])
    kolom_tgl = st.selectbox("Kolom tanggal", kolom, index=kolom.index(tebak_tgl))
    kolom_nilai = st.selectbox("Kolom nilai (level, belum di-log)", kolom, index=kolom.index(tebak_nilai))

validasi = eng.siapkan_seri(df_mentah, kolom_tgl, kolom_nilai)
for p in validasi.peringatan:
    st.warning(p)
if not validasi.ok:
    for g in validasi.galat:
        st.error(g)
    st.stop()

y = validasi.seri
daftar_periode = [str(p) for p in y.index]


def _pilih(label, default, opsi, key):
    idx = opsi.index(default) if default in opsi else 0
    return st.selectbox(label, opsi, index=idx, key=key)


with st.sidebar:
    st.header("2. Periode")
    awal_latih = _pilih("Awal data latih", "2021-01", daftar_periode, "awal")
    akhir_latih = _pilih("Akhir data latih", "2025-06", daftar_periode, "akhir_latih")
    opsi_uji = [p for p in daftar_periode if p > akhir_latih] or [akhir_latih]
    akhir_uji = _pilih("Akhir data uji (backtest)", "2026-06", opsi_uji, "akhir_uji")

    st.header("3. Model")
    c1, c2, c3 = st.columns(3)
    p = c1.number_input("p", 0, 5, 0)
    d = c2.number_input("d", 0, 2, 0)
    q = c3.number_input("q", 0, 5, 0)
    c1, c2, c3 = st.columns(3)
    P = c1.number_input("P", 0, 2, 1)
    D = c2.number_input("D", 0, 1, 0)
    Q = c3.number_input("Q", 0, 2, 1)
    s = st.number_input("Periode musiman (s)", 1, 24, 12)
    konstanta = st.checkbox("Konstanta C (rata-rata, seperti EViews)", value=True)
    with st.expander("Opsi lanjutan"):
        paksa_stasioner = st.checkbox(
            "Paksa stasioner", value=False,
            help="EViews tidak memaksa. Aktifkan bila estimasi lari ke AR = 1.")
        paksa_invertible = st.checkbox("Paksa invertible", value=False)
        koreksi_bias = st.checkbox(
            "Koreksi bias saat exp()", value=False,
            help="Mengalikan exp(proyeksi log) dengan exp(σ²/2). EViews tidak melakukannya secara default.")

    st.header("4. Proyeksi")
    langkah = st.slider("Horizon (bulan)", 1, 36, 12)
    tingkat = st.select_slider("Tingkat kepercayaan", [80, 90, 95, 99], value=95)
    akhir_data_proyeksi = _pilih("Estimasi ulang dengan data s.d.", daftar_periode[-1], daftar_periode, "akhir_proj")
    jalan = st.button("Jalankan model", type="primary", use_container_width=True)

spek = eng.SpesifikasiModel(
    order=(int(p), int(d), int(q)),
    seasonal_order=(int(P), int(D), int(Q), int(s)),
    konstanta=konstanta,
    paksa_stasioner=paksa_stasioner,
    paksa_invertible=paksa_invertible,
)
alpha = 1 - tingkat / 100

if jalan:
    with st.spinner("Mengestimasi model..."):
        try:
            bt = eng.backtest(y, spek, awal_latih, akhir_latih, akhir_uji, alpha, koreksi_bias)
            y_proj = eng.potong(y, awal_latih, akhir_data_proyeksi)
            hasil_proj = eng.estimasi(np.log(y_proj), spek)
            st.session_state["hasil"] = {
                "spek": spek,
                "bt": bt,
                "diag": eng.diagnostik(bt["hasil"], spek),
                "stat": eng.statistik_ringkas(bt["hasil"], np.log(eng.potong(y, awal_latih, akhir_latih))),
                "y_proj": y_proj,
                "ramalan": eng.proyeksi(hasil_proj, langkah, spek, alpha, koreksi_bias),
                "diag_proj": eng.diagnostik(hasil_proj, spek),
            }
        except Exception as e:
            st.error(f"Estimasi gagal: {e}")

# ---------------------------------------------------------------------------
# Tampilan
# ---------------------------------------------------------------------------
tab_data, tab_est, tab_diag, tab_bt, tab_proj = st.tabs(
    ["Data", "Estimasi", "Diagnostik", "Backtest", "Proyeksi"])

with tab_data:
    gambar = go.Figure()
    gambar.add_scatter(x=y.index.to_timestamp(), y=y, name="Outflow", mode="lines+markers")
    gambar.update_layout(height=380, margin=dict(t=30, b=10), yaxis_title="Rupiah")
    st.plotly_chart(gambar, use_container_width=True)
    c1, c2 = st.columns(2)
    c1.metric("Jumlah bulan", int(y.notna().sum()))
    c2.metric("Rentang", f"{y.index.min()} – {y.index.max()}")
    st.dataframe(y.to_frame("outflow").assign(log_outflow=np.log(y)).rename(index=str), use_container_width=True)

hasil_ss = st.session_state.get("hasil")
if hasil_ss is None:
    for t in (tab_est, tab_diag, tab_bt, tab_proj):
        t.info("Atur pengaturan di samping, lalu klik **Jalankan model**.")
    st.stop()

bt, diag = hasil_ss["bt"], hasil_ss["diag"]
label = hasil_ss["spek"].label()

with tab_est:
    st.subheader(f"{label} — data latih {awal_latih} s.d. {akhir_latih}")
    for pesan in diag["peringatan"]:
        st.warning(pesan)
    st.dataframe(eng.tabel_koefisien(bt["hasil"]).style.format("{:.6f}"), use_container_width=True)
    st.dataframe(hasil_ss["stat"].to_frame("Nilai").style.format("{:.6f}"), use_container_width=True)
    st.caption("Kriteria informasi dihitung per observasi seperti EViews. Kovarians: outer product of gradients.")

with tab_diag:
    c1, c2 = st.columns(2)
    with c1:
        st.markdown("**Ljung-Box (residual)** — p > 0,05 berarti tidak ada autokorelasi tersisa")
        st.dataframe(diag["ljung_box"], use_container_width=True)
        jb = diag["jarque_bera"]
        st.markdown(f"**Jarque-Bera**: statistik = {jb['statistik']:.3f}, p = {jb['p_value']:.4f} "
                    f"({'normal' if jb['p_value'] > 0.05 else 'tidak normal'})")
        st.markdown("**Modulus akar invers** (harus < 1)")
        st.write({"AR": np.round(diag["modulus_akar_ar_invers"].max(), 4) if len(diag["modulus_akar_ar_invers"]) else "-",
                  "MA": np.round(diag["modulus_akar_ma_invers"].max(), 4) if len(diag["modulus_akar_ma_invers"]) else "-"})
    with c2:
        acf_df = diag["acf"]
        g = go.Figure()
        g.add_bar(x=acf_df["lag"], y=acf_df["acf"], name="ACF residual")
        for tanda in (1, -1):
            g.add_hline(y=tanda * diag["batas_acf"], line_dash="dash", line_color="gray")
        g.update_layout(height=320, margin=dict(t=30, b=10), title="ACF residual")
        st.plotly_chart(g, use_container_width=True)

with tab_bt:
    tabel = bt["tabel"]
    x = tabel.index.to_timestamp()
    g = go.Figure()
    g.add_scatter(x=x, y=tabel["batas_atas"], line=dict(width=0), showlegend=False, hoverinfo="skip")
    g.add_scatter(x=x, y=tabel["batas_bawah"], fill="tonexty", line=dict(width=0),
                  fillcolor="rgba(31,119,180,0.15)", name=f"Selang {tingkat}%")
    g.add_scatter(x=x, y=tabel["aktual"], name="Aktual", mode="lines+markers", line=dict(color="black"))
    g.add_scatter(x=x, y=tabel[label], name=label, mode="lines+markers")
    g.add_scatter(x=x, y=tabel["Naif musiman"], name="Naif musiman", line=dict(dash="dot"))
    g.add_scatter(x=x, y=tabel["Rata-rata per bulan"], name="Rata-rata per bulan", line=dict(dash="dash"))
    g.update_layout(height=420, margin=dict(t=30, b=10), yaxis_title="Rupiah")
    st.plotly_chart(g, use_container_width=True)
    st.markdown("**Ukuran akurasi** (proyeksi dinamis dari akhir data latih)")
    st.dataframe(bt["akurasi"].style.format("{:,.2f}"), use_container_width=True)
    st.dataframe(tabel.rename(index=str).style.format("{:,.0f}"), use_container_width=True)

with tab_proj:
    ramalan, y_proj = hasil_ss["ramalan"], hasil_ss["y_proj"]
    for pesan in hasil_ss["diag_proj"]["peringatan"]:
        st.warning(pesan)
    g = go.Figure()
    g.add_scatter(x=y_proj.index.to_timestamp(), y=y_proj, name="Historis", line=dict(color="black"))
    xr = ramalan.index.to_timestamp()
    g.add_scatter(x=xr, y=ramalan["batas_atas"], line=dict(width=0), showlegend=False, hoverinfo="skip")
    g.add_scatter(x=xr, y=ramalan["batas_bawah"], fill="tonexty", line=dict(width=0),
                  fillcolor="rgba(31,119,180,0.15)", name=f"Selang {tingkat}%")
    g.add_scatter(x=xr, y=ramalan["proyeksi"], name="Proyeksi", mode="lines+markers")
    g.update_layout(height=420, margin=dict(t=30, b=10), yaxis_title="Rupiah")
    st.plotly_chart(g, use_container_width=True)

    keluaran = ramalan[["proyeksi", "batas_bawah", "batas_atas"]].copy()
    keluaran.index = keluaran.index.astype(str)
    keluaran.index.name = "periode"
    st.dataframe(keluaran.style.format("{:,.0f}"), use_container_width=True)

    c1, c2 = st.columns(2)
    c1.download_button("Unduh CSV", keluaran.to_csv().encode("utf-8"), "proyeksi_outflow.csv", "text/csv")
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as w:
        keluaran.to_excel(w, sheet_name="proyeksi")
        eng.tabel_koefisien(bt["hasil"]).to_excel(w, sheet_name="koefisien")
        bt["akurasi"].to_excel(w, sheet_name="akurasi_backtest")
    c2.download_button("Unduh Excel", buf.getvalue(), "proyeksi_outflow.xlsx",
                       "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")

In [ ]:
# Unduh cloudflared, jalankan Streamlit + tunnel di latar belakang, lalu tampilkan tautannya.
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup streamlit run app.py --server.port 8501 --server.headless true &> streamlit.log &
!nohup ./cloudflared tunnel --url http://localhost:8501 &> cloudflared.log &
!sleep 10 && grep -o "https://[a-z0-9-]*\.trycloudflare\.com" cloudflared.log | head -1